In [ ]:
import io, datetime, requests, numpy as np, pandas as pd, ipywidgets as widgets
from zoneinfo import ZoneInfo
from IPython.display import display

SEASON = 2026
HEADERS = {"User-Agent": "Mozilla/5.0"}

t_url = f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/teams.csv"
g_url = f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/goalies.csv"
teams_df = pd.read_csv(io.StringIO(requests.get(t_url, headers=HEADERS, timeout=20).text))
goalies_df = pd.read_csv(io.StringIO(requests.get(g_url, headers=HEADERS, timeout=20).text))

MP_MAP = {"LAK": "L.A", "NJD": "N.J", "SJS": "S.J", "TBL": "T.B"}
def to_mp(abbr):
    return MP_MAP.get(abbr, abbr) if abbr not in teams_df["team"].values else abbr

lg_5v5 = teams_df[teams_df["situation"] == "5on5"]
LG_XG5 = lg_5v5["xGoalsFor"].sum() / (lg_5v5["iceTime"].sum() / 3600.0)
lg_pp = teams_df[teams_df["situation"] == "5on4"]
LG_XGPP = lg_pp["xGoalsFor"].sum() / (lg_pp["iceTime"].sum() / 3600.0)

def get_team(abbr):
    mp = to_mp(abbr)
    t = teams_df[teams_df["team"] == mp]
    s5, spp, spk, sall = [t[t["situation"] == s].iloc[0] for s in ("5on5", "5on4", "4on5", "all")]
    toi5, toipp, toipk, toiall = [max(x["iceTime"]/3600.0, 0.05) for x in (s5, spp, spk, sall)]
    xgf_c = "scoreVenueAdjustedxGoalsFor" if "scoreVenueAdjustedxGoalsFor" in s5.index else "xGoalsFor"
    xga_c = "scoreVenueAdjustedxGoalsAgainst" if "scoreVenueAdjustedxGoalsAgainst" in s5.index else "xGoalsAgainst"
    hdcf, hdca = s5["highDangerShotsFor"]/toi5, s5["highDangerShotsAgainst"]/toi5
    return {
        "xgf60": s5[xgf_c]/toi5, "xga60": s5[xga_c]/toi5,
        "cf60": s5["shotAttemptsFor"]/toi5, "ca60": s5["shotAttemptsAgainst"]/toi5,
        "hdcf_pct": (hdcf/max(hdcf+hdca, 0.1))*100,
        "pp_xgf60": spp["xGoalsFor"]/toipp, "pk_xga60": spk["xGoalsAgainst"]/toipk,
        "drawn60": sall["penaltiesAgainst"]/toiall, "taken60": sall["penaltiesFor"]/toiall
    }

def get_goalie_cal(abbr, goalie_name=None):
    mp = to_mp(abbr)
    team_g = goalies_df[(goalies_df["situation"] == "all") & (goalies_df["team"] == mp)].sort_values("games_played", ascending=False)
    starter_name = team_g.iloc[0]["name"]
    target_name = goalie_name if goalie_name else starter_name
    g = team_g[team_g["name"] == target_name].iloc[0]
    gp = int(g["games_played"])
    toi = max(g["icetime"] / 3600.0, 0.1)
    raw_gsax60 = (g["xGoals"] - g["goals"]) / toi
    is_backup = (target_name != starter_name)
    adj60 = (((raw_gsax60 * toi) + (-0.18 * 20.0)) / (toi + 20.0) - 0.10) if is_backup else ((raw_gsax60 * toi) / (toi + 15.0))
    return {"name": f"{target_name} ({gp} GP)", "adj60": adj60, "is_backup": is_backup, "all_goalies": team_g["name"].tolist()}

est_now = datetime.datetime.now(ZoneInfo("America/New_York"))
today_str = est_now.strftime("%Y-%m-%d")
yest_str = (est_now - datetime.timedelta(days=1)).strftime("%Y-%m-%d")

sched_today = requests.get(f"https://api-web.nhle.com/v1/schedule/{today_str}", timeout=15).json()
sched_yest = requests.get(f"https://api-web.nhle.com/v1/schedule/{yest_str}", timeout=15).json()

b2b_teams = {g[k]["abbrev"] for d in sched_yest.get("gameWeek", []) if d.get("date") == yest_str for g in d.get("games", []) for k in ("awayTeam", "homeTeam")}
todays_games = [
    {"away": g["awayTeam"]["abbrev"], "home": g["homeTeam"]["abbrev"],
     "time": datetime.datetime.fromisoformat(g["startTimeUTC"].replace("Z", "+00:00")).astimezone(ZoneInfo("America/New_York")).strftime("%-I:%M %p ET")}
    for d in sched_today.get("gameWeek", []) if d.get("date") == today_str for g in d.get("games", [])
]

goalie_overrides = {}
slate_teams = sorted({g["away"] for g in todays_games} | {g["home"] for g in todays_games})
team_select = widgets.Dropdown(options=slate_teams, description="Team:")
goalie_select = widgets.Dropdown(description="Goalie:")
swap_btn = widgets.Button(description="Swap & Re-Sim", button_style="info")
html_board = widgets.HTML()

def on_team_change(change):
    if change["new"]:
        goalie_select.options = get_goalie_cal(change["new"])["all_goalies"]
team_select.observe(on_team_change, names="value")
if team_select.value:
    on_team_change({"new": team_select.value})

def build_slate_html():
    cards = [f"<div style='color:#94a3b8;font-size:12px;margin-bottom:8px;font-family:sans-serif;'>📅 <b>{today_str}</b> ({len(todays_games)} Games) &nbsp;|&nbsp; Auto-B2B: <b>{', '.join(sorted(b2b_teams)) if b2b_teams else 'None'}</b></div>"]
    rng = np.random.default_rng(42)
    for g in todays_games:
        aa, ha = g["away"], g["home"]
        a, h = get_team(aa), get_team(ha)
        ag, hg = get_goalie_cal(aa, goalie_overrides.get(aa)), get_goalie_cal(ha, goalie_overrides.get(ha))
        a_b2b, h_b2b = (aa in b2b_teams), (ha in b2b_teams)
        app_m = np.clip(5.2 + (a["drawn60"]-3.3)*0.65 + (h["taken60"]-3.3)*0.65, 2.5, 9.5)
        hpp_m = np.clip(5.2 + (h["drawn60"]-3.3)*0.65 + (a["taken60"]-3.3)*0.65, 2.5, 9.5)
        ev_m = max(60.0 - app_m - hpp_m, 42.0)
        a5_rate = a["xgf60"] * (h["xga60"]/LG_XG5) * 0.965 * (0.94 if a_b2b else 1.0) * (1.04 if h_b2b else 1.0)
        h5_rate = h["xgf60"] * (a["xga60"]/LG_XG5) * 1.035 * (0.94 if h_b2b else 1.0) * (1.04 if a_b2b else 1.0)
        a_5v5, h_5v5 = a5_rate*(ev_m/60.0), h5_rate*(ev_m/60.0)
        a_pp, h_pp = (a["pp_xgf60"]*(h["pk_xga60"]/LG_XGPP))*(app_m/60.0), (h["pp_xgf60"]*(a["pk_xga60"]/LG_XGPP))*(hpp_m/60.0)
        alam, hlam = max(a_5v5 + a_pp - hg["adj60"] + 0.16, 0.8), max(h_5v5 + h_pp - ag["adj60"] + 0.18, 0.8)
        ar, hr = rng.poisson(rng.gamma(18.0, alam/18.0, 10000)), rng.poisson(rng.gamma(18.0, hlam/18.0, 10000))
        ties = ar == hr
        af, hf = ar.copy(), hr.copy()
        if ties.sum() > 0:
            aot = rng.random(ties.sum()) < (0.60*(a5_rate/(a5_rate+h5_rate)) + 0.20)
            idx = np.where(ties)[0]
            af[idx[aot]] += 1
            hf[idx[~aot]] += 1
        tot = af + hf
        comb_cf = a["cf60"] + h["cf60"]
        tag = "<span style='background:#7f1d1d;color:#fca5a5;padding:2px 6px;border-radius:4px;font-size:11px'>🔥 TRACK MEET</span>" if (comb_cf >= 122 and tot.mean() >= 6.2) else ("<span style='background:#1e3a8a;color:#93c5fd;padding:2px 6px;border-radius:4px;font-size:11px'>🧱 LOW-EVENT</span>" if (comb_cf <= 112 and tot.mean() <= 5.6) else "<span style='background:#334155;color:#cbd5e1;padding:2px 6px;border-radius:4px;font-size:11px'>⚖️ BALANCED</span>")
        a_badge = " <span style='background:#b45309;color:#fde68a;padding:1px 4px;border-radius:3px;font-size:10px'>B2B</span>" if a_b2b else ""
        h_badge = " <span style='background:#b45309;color:#fde68a;padding:1px 4px;border-radius:3px;font-size:10px'>B2B</span>" if h_b2b else ""
        ag_badge = " <span style='background:#991b1b;color:#fecaca;padding:1px 4px;border-radius:3px;font-size:9px'>BACKUP</span>" if ag["is_backup"] else ""
        hg_badge = " <span style='background:#991b1b;color:#fecaca;padding:1px 4px;border-radius:3px;font-size:9px'>BACKUP</span>" if hg["is_backup"] else ""
        cards.append(f"<div style='background:#0f172a;border:1px solid #334155;border-radius:10px;padding:12px;margin-bottom:12px;font-family:-apple-system,sans-serif;color:#f8fafc;'><div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:6px;'><div style='font-size:16px;font-weight:800;'>{aa}{a_badge} @ {ha}{h_badge} <span style='font-size:11px;color:#94a3b8;font-weight:400;'>({g['time']})</span></div><div>{tag}</div></div><div style='display:flex;justify-content:space-between;background:#1e293b;padding:8px 10px;border-radius:6px;margin-bottom:8px;font-size:13px;'><div><b>Proj:</b> {aa} {af.mean():.2f} — {ha} {hf.mean():.2f}</div><div><b>Win:</b> {aa} {(af>hf).mean()*100:.0f}% | {ha} {(hf>af).mean()*100:.0f}%</div><div><b>Tot:</b> {tot.mean():.2f} (O5.5: {(tot>5.5).mean()*100:.0f}%)</div></div><table style='width:100%;border-collapse:collapse;font-size:12px;color:#cbd5e1;'><tr style='border-bottom:1px solid #334155;color:#94a3b8;'><th style='text-align:left;padding:3px;'>Per-Game Breakdown</th><th>{aa}</th><th>{ha}</th></tr><tr><td style='padding:3px;'>5v5 Expected Goals ({ev_m:.0f}m)</td><td style='text-align:center;'><b>{a_5v5:.2f}</b></td><td style='text-align:center;'><b>{h_5v5:.2f}</b></td></tr><tr style='background:#1e293b60;'><td style='padding:3px;'>Power Play xG (PP Mins)</td><td style='text-align:center;'><b>{a_pp:.2f}</b> ({app_m:.1f}m)</td><td style='text-align:center;'><b>{h_pp:.2f}</b> ({hpp_m:.1f}m)</td></tr><tr><td style='padding:3px;'>Pace (CF/60) & Slot (HDCF%)</td><td style='text-align:center;'>{a['cf60']:.1f} | {a['hdcf_pct']:.0f}%</td><td style='text-align:center;'>{h['cf60']:.1f} | {h['hdcf_pct']:.0f}%</td></tr><tr style='background:#1e293b60;'><td style='padding:3px;'>Goalie Saved (GSAx/Game)</td><td style='text-align:center;'><b>{ag['name']}</b>{ag_badge} ({ag['adj60']:+.2f})</td><td style='text-align:center;'><b>{hg['name']}</b>{hg_badge} ({hg['adj60']:+.2f})</td></tr></table></div>")
    return "".join(cards)

swap_btn.on_click(lambda b: (goalie_overrides.update({team_select.value: goalie_select.value}), setattr(html_board, "value", build_slate_html())))
html_board.value = build_slate_html()
display(widgets.VBox([widgets.HBox([team_select, goalie_select, swap_btn]), html_board]))

from google.colab import files



In [ ]:
import io, datetime, requests, numpy as np, pandas as pd, ipywidgets as widgets
from zoneinfo import ZoneInfo
from IPython.display import display

SEASON = 2026
HEADERS = {"User-Agent": "Mozilla/5.0"}

teams_df = pd.read_csv(io.StringIO(requests.get(f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/teams.csv", headers=HEADERS, timeout=20).text))
goalies_df = pd.read_csv(io.StringIO(requests.get(f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/goalies.csv", headers=HEADERS, timeout=20).text))
skaters_df = pd.read_csv(io.StringIO(requests.get(f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/skaters.csv", headers=HEADERS, timeout=20).text))

MP_MAP = {"LAK": "L.A", "NJD": "N.J", "SJS": "S.J", "TBL": "T.B"}
def to_mp(abbr):
    return MP_MAP.get(abbr, abbr) if abbr not in teams_df["team"].values else abbr

lg_5v5 = teams_df[teams_df["situation"] == "5on5"]
LG_XG5 = lg_5v5["xGoalsFor"].sum() / (lg_5v5["iceTime"].sum() / 3600.0)
LG_CA60 = lg_5v5["shotAttemptsAgainst"].sum() / (lg_5v5["iceTime"].sum() / 3600.0)
lg_pp = teams_df[teams_df["situation"] == "5on4"]
LG_XGPP = lg_pp["xGoalsFor"].sum() / (lg_pp["iceTime"].sum() / 3600.0)

est_now = datetime.datetime.now(ZoneInfo("America/New_York"))
today_str = est_now.strftime("%Y-%m-%d")
yest_str = (est_now - datetime.timedelta(days=1)).strftime("%Y-%m-%d")

sched_today = requests.get(f"https://api-web.nhle.com/v1/schedule/{today_str}", timeout=15).json()
sched_yest = requests.get(f"https://api-web.nhle.com/v1/schedule/{yest_str}", timeout=15).json()

b2b_teams = {g[k]["abbrev"] for d in sched_yest.get("gameWeek", []) if d.get("date") == yest_str for g in d.get("games", []) for k in ("awayTeam", "homeTeam")}
todays_games = [
    {"away": g["awayTeam"]["abbrev"], "home": g["homeTeam"]["abbrev"],
     "time": datetime.datetime.fromisoformat(g["startTimeUTC"].replace("Z", "+00:00")).astimezone(ZoneInfo("America/New_York")).strftime("%-I:%M %p ET")}
    for d in sched_today.get("gameWeek", []) if d.get("date") == today_str for g in d.get("games", [])
]

game_labels = [f"{g['away']} @ {g['home']} ({g['time']})" for g in todays_games]
game_dd = widgets.Dropdown(options=game_labels, description="Matchup:")
player_board = widgets.HTML()

def project_team_skaters(team_abbr, opp_abbr, is_home):
    mp_team, mp_opp = to_mp(team_abbr), to_mp(opp_abbr)
    t_opp = teams_df[teams_df["team"] == mp_opp]
    s5_opp = t_opp[t_opp["situation"] == "5on5"].iloc[0]
    spk_opp = t_opp[t_opp["situation"] == "4on5"].iloc[0]
    toi5_opp = max(s5_opp["iceTime"]/3600.0, 0.1)
    toipk_opp = max(spk_opp["iceTime"]/3600.0, 0.05)
    xga_c = "scoreVenueAdjustedxGoalsAgainst" if "scoreVenueAdjustedxGoalsAgainst" in s5_opp.index else "xGoalsAgainst"

    opp_ca60 = s5_opp["shotAttemptsAgainst"] / toi5_opp
    opp_xga60 = s5_opp[xga_c] / toi5_opp
    opp_pk60 = spk_opp["xGoalsAgainst"] / toipk_opp

    opp_g = goalies_df[(goalies_df["situation"] == "all") & (goalies_df["team"] == mp_opp)].sort_values("games_played", ascending=False).iloc[0]
    g_toi = max(opp_g["icetime"] / 3600.0, 0.1)
    opp_gsax60 = (((opp_g["xGoals"] - opp_g["goals"]) / g_toi) * g_toi) / (g_toi + 15.0)

    is_b2b, opp_b2b = (team_abbr in b2b_teams), (opp_abbr in b2b_teams)
    pace_mult = (opp_ca60 / LG_CA60) * (1.035 if is_home else 0.965) * (0.95 if is_b2b else 1.0)
    xg_mult = (opp_xga60 / LG_XG5) * (1.035 if is_home else 0.965) * (0.94 if is_b2b else 1.0) * (1.04 if opp_b2b else 1.0)
    pp_mult = opp_pk60 / LG_XGPP

    pool = skaters_df[(skaters_df["team"] == mp_team) & (skaters_df["games_played"] >= 5)]
    s_all = pool[pool["situation"] == "all"]
    s_5v5 = pool[pool["situation"] == "5on5"].set_index("playerId")
    s_pp = pool[pool["situation"] == "5on4"].set_index("playerId")

    rows = []
    for _, p in s_all.iterrows():
        pid, gp = p["playerId"], max(int(p["games_played"]), 1)
        toi_pg = (p["icetime"] / 60.0) / gp
        p5 = s_5v5.loc[pid] if pid in s_5v5.index else None
        ppp = s_pp.loc[pid] if pid in s_pp.index else None
        ev_pg = ((p5["icetime"] / 60.0) / gp) if p5 is not None else toi_pg * 0.85
        pp_pg = ((ppp["icetime"] / 60.0) / gp) if ppp is not None else 0.0

        proj_sog = (p["I_F_shotsOnGoal"] / max(p["icetime"] / 3600.0, 0.1)) * (toi_pg / 60.0) * pace_mult
        ixg_5v5 = (p5["I_F_xGoals"] / max(p5["icetime"] / 3600.0, 0.1)) if p5 is not None else 0.0
        ixg_pp = (ppp["I_F_xGoals"] / max(ppp["icetime"] / 3600.0, 0.05)) if (ppp is not None and ppp["icetime"] > 60) else 0.0
        proj_g = ((ixg_5v5 * (ev_pg / 60.0) * xg_mult) + (ixg_pp * (pp_pg / 60.0) * pp_mult)) * max(1.0 - (opp_gsax60 / 3.0), 0.75)
        proj_a = ((p["I_F_primaryAssists"] + p["I_F_secondaryAssists"]) / max(p["icetime"] / 3600.0, 0.1)) * (toi_pg / 60.0) * xg_mult
        proj_pts = proj_g + proj_a
        rows.append({"name": p["name"], "pos": p["position"], "toi": toi_pg, "pp": pp_pg, "sog": proj_sog, "xg": proj_g, "atg": (1 - np.exp(-proj_g))*100, "pts": proj_pts, "pt_p": (1 - np.exp(-proj_pts))*100})
    return opp_g["name"], pd.DataFrame(rows).sort_values("pts", ascending=False).head(8)

def render_players(change=None):
    g = todays_games[game_labels.index(game_dd.value)]
    aa, ha = g["away"], g["home"]
    hg_name, df_a = project_team_skaters(aa, ha, False)
    ag_name, df_h = project_team_skaters(ha, aa, True)
    def tbl(title, df):
        trs = "".join([f"<tr style='border-bottom:1px solid #334155;'><td style='padding:5px;text-align:left;'><b>{r['name']}</b> <span style='color:#94a3b8;font-size:10px;'>({r['pos']})</span></td><td style='text-align:center;'>{r['toi']:.1f}m <span style='color:#38bdf8;font-size:10px;'>({r['pp']:.1f} PP)</span></td><td style='text-align:center;'><b>{r['sog']:.2f}</b></td><td style='text-align:center;'>{r['xg']:.2f} ({r['atg']:.0f}%)</td><td style='text-align:center;'><b>{r['pts']:.2f}</b> ({r['pt_p']:.0f}%)</td></tr>" for _, r in df.iterrows()])
        return f"<div style='background:#0f172a;border:1px solid #334155;border-radius:10px;padding:10px;margin-bottom:12px;font-family:-apple-system,sans-serif;color:#f8fafc;'><div style='font-size:14px;font-weight:800;margin-bottom:6px;color:#38bdf8;'>{title}</div><table style='width:100%;border-collapse:collapse;font-size:12px;color:#cbd5e1;'><tr style='background:#1e293b;color:#94a3b8;'><th style='padding:5px;text-align:left;'>Player</th><th>TOI (PP)</th><th>Proj SOG</th><th>xG (ATG%)</th><th>Proj Pts (1+%)</th></tr>{trs}</table></div>"
    player_board.value = tbl(f"{aa} Skaters (vs {hg_name})", df_a) + tbl(f"{ha} Skaters (vs {ag_name})", df_h)

game_dd.observe(render_players, names="value")
render_players()
display(widgets.VBox([game_dd, player_board]))




In [ ]:
import io, datetime, requests, numpy as np, pandas as pd, ipywidgets as widgets
from zoneinfo import ZoneInfo
from IPython.display import display

SEASON = 2026
HEADERS = {"User-Agent": "Mozilla/5.0"}

# 1. Load MoneyPuck Data & Today's Schedule (Self-Contained)
teams_df = pd.read_csv(io.StringIO(requests.get(f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/teams.csv", headers=HEADERS, timeout=20).text))
goalies_df = pd.read_csv(io.StringIO(requests.get(f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/goalies.csv", headers=HEADERS, timeout=20).text))
skaters_df = pd.read_csv(io.StringIO(requests.get(f"https://moneypuck.com/moneypuck/playerData/seasonSummary/{SEASON}/regular/skaters.csv", headers=HEADERS, timeout=20).text))

MP_MAP = {"LAK": "L.A", "NJD": "N.J", "SJS": "S.J", "TBL": "T.B"}
def to_mp(abbr):
    return MP_MAP.get(abbr, abbr) if abbr not in teams_df["team"].values else abbr

lg_5v5 = teams_df[teams_df["situation"] == "5on5"]
LG_XG5 = lg_5v5["xGoalsFor"].sum() / (lg_5v5["iceTime"].sum() / 3600.0)
LG_CA60 = lg_5v5["shotAttemptsAgainst"].sum() / (lg_5v5["iceTime"].sum() / 3600.0)
lg_pp = teams_df[teams_df["situation"] == "5on4"]
LG_XGPP = lg_pp["xGoalsFor"].sum() / (lg_pp["iceTime"].sum() / 3600.0)

est_now = datetime.datetime.now(ZoneInfo("America/New_York"))
today_str = est_now.strftime("%Y-%m-%d")
yest_str = (est_now - datetime.timedelta(days=1)).strftime("%Y-%m-%d")

sched_today = requests.get(f"https://api-web.nhle.com/v1/schedule/{today_str}", timeout=15).json()
sched_yest = requests.get(f"https://api-web.nhle.com/v1/schedule/{yest_str}", timeout=15).json()

b2b_teams = {g[k]["abbrev"] for d in sched_yest.get("gameWeek", []) if d.get("date") == yest_str for g in d.get("games", []) for k in ("awayTeam", "homeTeam")}
all_games = [
    {"id": g["id"], "away": g["awayTeam"]["abbrev"], "home": g["homeTeam"]["abbrev"], "state": g.get("gameState", "FUT")}
    for d in sched_today.get("gameWeek", []) if d.get("date") == today_str for g in d.get("games", [])
]

labels = [f"{g['away']} @ {g['home']} [{g['state']}]" for g in all_games]
backtest_dd = widgets.Dropdown(options=labels, description="Test Game:")
refresh_btn = widgets.Button(description="Refresh Box Score", button_style="success")
eval_board = widgets.HTML()

def run_backtest(b=None):
    g = all_games[labels.index(backtest_dd.value)]
    aa, ha, gid = g["away"], g["home"], g["id"]

    # Fetch live/final NHL API boxscore
    box = requests.get(f"https://api-web.nhle.com/v1/gamecenter/{gid}/boxscore", timeout=15).json()
    state = box.get("gameState", g["state"])
    act_a_goals = box.get("awayTeam", {}).get("score", 0)
    act_h_goals = box.get("homeTeam", {}).get("score", 0)
    act_a_sog = box.get("awayTeam", {}).get("sog", 0)
    act_h_sog = box.get("homeTeam", {}).get("sog", 0)

    # Run Team Model Projection
    def get_t(abbr):
        t = teams_df[teams_df["team"] == to_mp(abbr)]
        s5, spp, spk, sall = [t[t["situation"] == s].iloc[0] for s in ("5on5", "5on4", "4on5", "all")]
        toi5, toipp, toipk, toiall = [max(x["iceTime"]/3600.0, 0.05) for x in (s5, spp, spk, sall)]
        xgf_c = "scoreVenueAdjustedxGoalsFor" if "scoreVenueAdjustedxGoalsFor" in s5.index else "xGoalsFor"
        xga_c = "scoreVenueAdjustedxGoalsAgainst" if "scoreVenueAdjustedxGoalsAgainst" in s5.index else "xGoalsAgainst"
        return {"xgf60": s5[xgf_c]/toi5, "xga60": s5[xga_c]/toi5, "ca60": s5["shotAttemptsAgainst"]/toi5,
                "pp_xgf60": spp["xGoalsFor"]/toipp, "pk_xga60": spk["xGoalsAgainst"]/toipk,
                "drawn60": sall["penaltiesAgainst"]/toiall, "taken60": sall["penaltiesFor"]/toiall}

    def get_g(abbr):
        tg = goalies_df[(goalies_df["situation"] == "all") & (goalies_df["team"] == to_mp(abbr))].sort_values("games_played", ascending=False).iloc[0]
        toi = max(tg["icetime"]/3600.0, 0.1)
        return tg["name"], ((tg["xGoals"] - tg["goals"]) / toi) * (toi / (toi + 15.0))

    a, h = get_t(aa), get_t(ha)
    ag_name, ag_adj = get_g(aa)
    hg_name, hg_adj = get_g(ha)
    a_b2b, h_b2b = (aa in b2b_teams), (ha in b2b_teams)

    app_m = np.clip(5.2 + (a["drawn60"]-3.3)*0.65 + (h["taken60"]-3.3)*0.65, 2.5, 9.5)
    hpp_m = np.clip(5.2 + (h["drawn60"]-3.3)*0.65 + (a["taken60"]-3.3)*0.65, 2.5, 9.5)
    ev_m = max(60.0 - app_m - hpp_m, 42.0)

    a5 = a["xgf60"] * (h["xga60"]/LG_XG5) * 0.965 * (0.94 if a_b2b else 1.0) * (1.04 if h_b2b else 1.0)
    h5 = h["xgf60"] * (a["xga60"]/LG_XG5) * 1.035 * (0.94 if h_b2b else 1.0) * (1.04 if a_b2b else 1.0)
    alam = max(a5*(ev_m/60.0) + (a["pp_xgf60"]*(h["pk_xga60"]/LG_XGPP))*(app_m/60.0) - hg_adj + 0.16, 0.8)
    hlam = max(h5*(ev_m/60.0) + (h["pp_xgf60"]*(a["pk_xga60"]/LG_XGPP))*(hpp_m/60.0) - ag_adj + 0.18, 0.8)

    rng = np.random.default_rng(42)
    ar, hr = rng.poisson(rng.gamma(18.0, alam/18.0, 10000)), rng.poisson(rng.gamma(18.0, hlam/18.0, 10000))
    ties = ar == hr
    af, hf = ar.copy(), hr.copy()
    if ties.sum() > 0:
        aot = rng.random(ties.sum()) < (0.60*(a5/(a5+h5)) + 0.20)
        idx = np.where(ties)[0]
        af[idx[aot]] += 1
        hf[idx[~aot]] += 1

    proj_a, proj_h = af.mean(), hf.mean()
    proj_tot = (af + hf).mean()
    act_tot = act_a_goals + act_h_goals
    pred_winner = aa if proj_a > proj_h else ha
    act_winner = aa if act_a_goals > act_h_goals else (ha if act_h_goals > act_a_goals else "TIE")
    win_hit = "✅ HIT" if pred_winner == act_winner else ("⏳ LIVE" if state not in ("FINAL", "OFF") else "❌ MISS")

    # Parse Actual Player Stats from NHL API Box Score
    actual_players = {}
    p_stats = box.get("playerByGameStats", {})
    for side in ("awayTeam", "homeTeam"):
        for group in ("forwards", "defense"):
            for p in p_stats.get(side, {}).get(group, []):
                pid = p.get("playerId")
                actual_players[pid] = {
                    "g": p.get("goals", 0),
                    "a": p.get("assists", 0),
                    "pts": p.get("points", 0),
                    "sog": p.get("sog", 0),
                }

    # Build Top Skater Comparison (Model Proj vs Actual)
    def skater_comp(abbr, opp_abbr, is_home, opp_g_adj):
        opp = get_t(opp_abbr)
        pace_m = (opp["ca60"] / LG_CA60) * (1.035 if is_home else 0.965)
        xg_m = (opp["xga60"] / LG_XG5) * (1.035 if is_home else 0.965)
        pp_m = opp["pk_xga60"] / LG_XGPP

        pool = skaters_df[(skaters_df["team"] == to_mp(abbr)) & (skaters_df["games_played"] >= 5)]
        s_all = pool[pool["situation"] == "all"]
        s_5 = pool[pool["situation"] == "5on5"].set_index("playerId")
        s_pp = pool[pool["situation"] == "5on4"].set_index("playerId")

        trs = []
        for _, p in s_all.iterrows():
            pid, gp = p["playerId"], max(int(p["games_played"]), 1)
            toi_pg = (p["icetime"] / 60.0) / gp
            p5 = s_5.loc[pid] if pid in s_5.index else None
            ppp = s_pp.loc[pid] if pid in s_pp.index else None
            ev_pg = ((p5["icetime"] / 60.0) / gp) if p5 is not None else toi_pg * 0.85
            pp_pg = ((ppp["icetime"] / 60.0) / gp) if ppp is not None else 0.0

            p_sog = (p["I_F_shotsOnGoal"] / max(p["icetime"] / 3600.0, 0.1)) * (toi_pg / 60.0) * pace_m
            ixg5 = (p5["I_F_xGoals"] / max(p5["icetime"] / 3600.0, 0.1)) if p5 is not None else 0.0
            ixgpp = (ppp["I_F_xGoals"] / max(ppp["icetime"] / 3600.0, 0.05)) if (ppp is not None and ppp["icetime"] > 60) else 0.0
            p_g = ((ixg5 * (ev_pg / 60.0) * xg_m) + (ixgpp * (pp_pg / 60.0) * pp_m)) * max(1.0 - (opp_g_adj / 3.0), 0.75)
            p_pts = p_g + ((p["I_F_primaryAssists"] + p["I_F_secondaryAssists"]) / max(p["icetime"] / 3600.0, 0.1)) * (toi_pg / 60.0) * xg_m
            trs.append((p_pts, pid, p["name"], p_sog, p_pts))

        trs.sort(reverse=True)
        html_rows = ""
        for _, pid, name, p_sog, p_pts in trs[:6]:
            act = actual_players.get(pid, {"g": 0, "a": 0, "pts": 0, "sog": 0})
            html_rows += f"<tr style='border-bottom:1px solid #334155;'><td style='padding:4px;text-align:left;'><b>{name}</b></td><td style='text-align:center;'>{p_sog:.1f} vs <b>{act['sog']}</b></td><td style='text-align:center;'>{p_pts:.2f} vs <b>{act['pts']}</b> ({act['g']}G {act['a']}A)</td></tr>"
        return html_rows

    a_rows = skater_comp(aa, ha, False, hg_adj)
    h_rows = skater_comp(ha, aa, True, ag_adj)

    eval_board.value = f"""
    <div style='background:#0f172a;border:1px solid #334155;border-radius:10px;padding:12px;font-family:-apple-system,sans-serif;color:#f8fafc;'>
        <div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:8px;'>
            <div style='font-size:16px;font-weight:800;'>BACKTEST: {aa} @ {ha} ({state})</div>
            <div style='background:#1e293b;padding:3px 8px;border-radius:5px;font-size:12px;font-weight:700;'>Winner Call: {win_hit}</div>
        </div>
        <table style='width:100%;border-collapse:collapse;font-size:13px;margin-bottom:12px;background:#1e293b;border-radius:6px;'>
            <tr style='color:#94a3b8;border-bottom:1px solid #334155;'><th style='padding:6px;text-align:left;'>Metric</th><th>Model Proj</th><th>Actual</th><th>Diff</th></tr>
            <tr><td style='padding:6px;'><b>{aa} Goals</b></td><td style='text-align:center;'>{proj_a:.2f}</td><td style='text-align:center;'><b>{act_a_goals}</b></td><td style='text-align:center;'>{act_a_goals - proj_a:+.2f}</td></tr>
            <tr><td style='padding:6px;'><b>{ha} Goals</b></td><td style='text-align:center;'>{proj_h:.2f}</td><td style='text-align:center;'><b>{act_h_goals}</b></td><td style='text-align:center;'>{act_h_goals - proj_h:+.2f}</td></tr>
            <tr style='border-top:1px solid #334155;'><td style='padding:6px;'><b>Total Goals</b></td><td style='text-align:center;'>{proj_tot:.2f}</td><td style='text-align:center;'><b>{act_tot}</b></td><td style='text-align:center;'>{act_tot - proj_tot:+.2f}</td></tr>
            <tr><td style='padding:6px;'><b>Shots on Goal</b></td><td style='text-align:center;'>Pace: {a['ca60']:.0f} / {h['ca60']:.0f}</td><td style='text-align:center;'><b>{act_a_sog} — {act_h_sog}</b></td><td style='text-align:center;'>{ha} +{act_h_sog - act_a_sog} SOG</td></tr>
        </table>
        <div style='font-size:13px;font-weight:700;color:#38bdf8;margin-bottom:4px;'>{aa} Top Skaters (Proj vs Actual)</div>
        <table style='width:100%;border-collapse:collapse;font-size:12px;color:#cbd5e1;margin-bottom:10px;'>
            <tr style='background:#1e293b;color:#94a3b8;'><th style='padding:4px;text-align:left;'>Player</th><th>SOG (Proj vs Act)</th><th>Points (Proj vs Act)</th></tr>{a_rows}
        </table>
        <div style='font-size:13px;font-weight:700;color:#38bdf8;margin-bottom:4px;'>{ha} Top Skaters (Proj vs Actual)</div>
        <table style='width:100%;border-collapse:collapse;font-size:12px;color:#cbd5e1;'>
            <tr style='background:#1e293b;color:#94a3b8;'><th style='padding:4px;text-align:left;'>Player</th><th>SOG (Proj vs Act)</th><th>Points (Proj vs Act)</th></tr>{h_rows}
        </table>
    </div>
    """

backtest_dd.observe(run_backtest, names="value")
refresh_btn.on_click(run_backtest)
run_backtest()
display(widgets.VBox([widgets.HBox([backtest_dd, refresh_btn]), eval_board]))
